# AF2RN — static + train-only observability gate
Menjalankan static audit lalu observability pada seluruh 1.665 train images. Tidak training, tidak membaca validation, dan test tidak boleh tersedia. Kirim hasil sebelum notebook training dibuat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,shutil,subprocess,sys,tarfile,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'; BRANCH='codex/af2-radially-normalized-angular-density'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],cwd=WORK,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
from coffee_detector.drive_project import require_project_artifact,resolve_drive_project_root
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'; D0_REL='experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt'
PROJECT=resolve_drive_project_root(required_relative_paths=(ARCHIVE_REL,D0_REL))
ARCHIVE=require_project_artifact(PROJECT,ARCHIVE_REL); D0=require_project_artifact(PROJECT,D0_REL)
DATA=WORK/'faruq-development-v3-grouped'
if not (DATA/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as stream: stream.extractall(WORK,filter='data')
assert (DATA/'train/images').is_dir() and (DATA/'faruq_grouped_summary.json').is_file()
assert not (DATA/'test').exists(),'STOP: test tidak boleh tersedia.'
OUTPUT=PROJECT/'experiments/faruq-v3-af2rn-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'; OBS=OUTPUT/'observability_train.json'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2_rn.audit import run_af2rn_static_audit
static=run_af2rn_static_audit(D0,STATIC,device='cuda:0')
print('STATIC DECISION:',static['decision']); print('PARAMETERS:',static['parameters']); print('GATES:',static['gates'])
assert static['decision']=='PASS','STOP: static audit FAIL; jangan observability/training.'


In [ ]:
from coffee_detector.af2_rn.observability import run_af2rn_observability_audit
obs=run_af2rn_observability_audit(DATA,DATA/'faruq_grouped_summary.json',STATIC,OBS,device='0',image_size=128,patches_per_image=16)
print('OBS DECISION:',obs['decision']); print('IMAGES:',obs['images'])
print('NONDEGENERATE:',obs['nondegenerate_fraction']); print('DIFFERS FROM AF2:',obs['different_from_af2_fraction'])
print('DISTRIBUTIONS:',obs['distributions']); print('RADIAL:',obs['radial_retention']); print('GATES:',obs['gates'])
print('TRAINING AUTHORIZED:',obs['training_authorized']); print('STATIC:',STATIC); print('OBS:',OBS)
print('Kirim output ini. Jangan training; notebook training dibuat hanya setelah PASS.')
